# D533 Introduction to Dataproc

**Dataproc runs Spark and Hadoop workloads on Google Cloud.** Use it to process files, join datasets, prepare analytical tables, and run existing PySpark pipelines.

Google documentation also uses the name **Managed Service for Apache Spark**. This notebook uses Dataproc, matching the familiar service name and `gcloud dataproc` commands.

The running example is a retail pipeline: read order files from GCS, remove invalid rows, calculate product sales, and write Parquet files back to GCS. Basic AWS knowledge is assumed; Amazon EMR is the closest starting comparison.

This is an introduction. Code blocks illustrate jobs submitted to a configured Spark environment; they are not local Python execution cells.

[Dataproc overview](https://docs.cloud.google.com/managed-spark/docs/concepts/clusters-overview)

## 1. Where Dataproc fits

```text
Application exports
        |
        v
GCS: raw CSV / JSON / Parquet
        |
        v
Dataproc: Spark transformations
        |
        v
GCS: curated Parquet
        |
        v
BigQuery: analytical tables and reports
```

**GCS stores the data. Dataproc supplies the processing environment.** BigQuery can consume the prepared data for analytics.

Use Dataproc for existing Spark applications, large joins, reusable PySpark transformations, or workloads that depend on the Hadoop ecosystem. Google manages cluster provisioning and service integration; you still choose resources, package dependencies, and write the transformation logic.

## 2. What a cluster contains

A cluster is a group of virtual machines working together. In a typical multi-node Dataproc cluster:

| Component | Responsibility |
|---|---|
| Master node | Runs coordination services, commonly including YARN ResourceManager and HDFS NameNode |
| Primary worker nodes | Run processing tasks and commonly store HDFS data |
| Secondary worker nodes | Add processing capacity; do not store HDFS data |
| YARN | Allocates cluster resources to applications |
| Spark | Executes distributed transformations |
| HDFS | Provides storage tied to the cluster |

```text
             Cluster coordination
                  Master
                     |
          +----------+----------+
          |                     |
       Worker 1              Worker 2
     Spark executors       Spark executors
          |                     |
          +------ GCS data -----+
```

A single-node cluster puts services on one VM and suits small demonstrations. Standard clusters distribute work across workers. High-availability configurations use multiple masters for supported coordination services; they do not make every application failure impossible.

[Cluster architecture and execution](https://docs.cloud.google.com/managed-spark/docs/concepts/jobs/life-of-a-job)

## 3. Driver, executors, partitions, and tasks

These are Spark concepts, separate from the VM roles above.

| Term | Meaning in an orders pipeline |
|---|---|
| Driver | Runs the application logic and coordinates execution |
| Executor | A process that executes tasks and holds application data |
| Partition | A portion of the dataset processed in parallel |
| Task | Work performed on one partition in a stage |
| Shuffle | Redistribution of data, often needed for joins or aggregation |

The driver asks executors to read partitions of the order data. A `groupBy("product")` can move matching product records together across executors. This movement is a shuffle and can require substantial network and disk activity.

Executors run on workers. Driver placement depends on the submission and deploy mode; do not assume the driver always runs on the master.

Adding workers can increase parallel capacity, but it does not fix a job that has too few partitions or one unusually large partition.

[Apache Spark cluster concepts](https://spark.apache.org/docs/latest/cluster-overview.html)

## 4. GCS and HDFS have different roles

Use GCS for input files and durable pipeline output. Spark accesses GCS through the Cloud Storage connector using `gs://` paths.

```text
gs://YOUR_BUCKET/raw/orders/
gs://YOUR_BUCKET/curated/product_sales/
gs://YOUR_BUCKET/scripts/product_sales.py
```

HDFS and local disks can support cluster-local work. Do not treat them as the only copy of business data: deleting the cluster can remove that storage.

A useful operating pattern is to keep data in GCS, create compute when needed, run the job, and remove the compute after output is written.

Spark output is usually a directory-like prefix containing several `part-...` files. Writing Parquet to `curated/product_sales/` does not produce one file named `product_sales.parquet`.

[Cloud Storage connector](https://docs.cloud.google.com/managed-spark/docs/concepts/connectors/cloud-storage)

## 5. Clusters and serverless Spark

| Choice | How you use it | Typical fit |
|---|---|---|
| Dataproc cluster | Create a cluster, submit jobs, manage its lifecycle | Shared environments, custom configuration, Hadoop ecosystem requirements |
| Serverless Spark | Submit a supported batch or start a supported interactive session | Spark work without maintaining a persistent cluster |

With a cluster, select VM types, disks, worker counts, software image, and optional components. Multiple jobs may share its resources.

With serverless Spark, Google manages the execution infrastructure and scaling. You still supply code, dependencies, identity, network configuration, and runtime settings. Serverless Spark is not a general-purpose Hadoop cluster.

**Example choice:** run an existing application that depends on a custom cluster setup on a cluster. Consider serverless for an independent PySpark batch that reads GCS and writes curated output.

[Serverless Spark overview](https://docs.cloud.google.com/managed-spark/docs/serverless-overview)

## 6. Decisions before creating a cluster

| Setting | What to decide |
|---|---|
| Project and region | Which project owns the resources, and where compute runs |
| Software image | Compatible Spark, Hadoop, Python, and connector versions |
| Master and worker machine types | CPU and memory for coordination and execution |
| Worker count | Initial parallel processing capacity |
| Disk size and type | Space and throughput for local work and shuffle |
| Service account | Identity used by jobs to access data |
| Network and subnet | Connectivity between nodes and to required services |
| Lifecycle | Persistent cluster or temporary cluster per workflow |

Keep processing and data in compatible locations. Account for project quotas and organization restrictions when choosing regions and machines.

A larger dataset does not automatically require a larger cluster: file format, joins, partitioning, and memory usage also matter. Start from the workload and adjust from its behavior.

## 7. Authentication: submitting a job versus running it

There are distinct identities:

1. **Submitter:** the user or automation calling the Dataproc API.
2. **Runtime service account:** the identity attached to the cluster VMs that accesses GCS and other data services.
3. **Google-managed service agent:** supports service operations behind the scenes.

The submitter needs permission to use Dataproc resources. Attaching a service account during cluster creation also requires permission to act as that account.

The runtime account needs Dataproc worker permissions and the required data access. For this example, it must read source objects and create output objects; overwrite workflows can also need delete permission.

A JSON key used locally in D532 does not automatically become the cluster's identity. On Dataproc, use its attached service account rather than copying that private key into the Spark script.

[Dataproc IAM](https://docs.cloud.google.com/managed-spark/docs/concepts/iam/iam)

## 8. Network access and Python dependencies

Cluster nodes need network communication with each other and access to required Google APIs. Private IP configurations commonly use Private Google Access for supported Google services. Downloading packages from external repositories may additionally require outbound access, such as Cloud NAT, or an internal package source.

The Ubuntu **dataengenv** environment is local to your machine. Its installed packages are not automatically installed on Dataproc workers.

For remote jobs, use the cluster's compatible Python runtime and distribute dependencies through supported mechanisms such as initialization actions, custom images, or job dependency files. Ensure required packages are available wherever the driver and executors run.

Installing `google-cloud-storage` in dataengenv gives local Python a storage client. It does not create a Spark cluster or configure a remote Spark runtime.

[Cluster networking](https://docs.cloud.google.com/managed-spark/docs/concepts/configuring-clusters/network)

## 9. Follow the life of a job

```text
Prepare script and input
          |
          v
Submit job to a cluster
          |
          v
Driver starts and builds the Spark plan
          |
          v
Executors read partitions and run tasks
          |
          v
Output is committed to GCS
          |
          v
Job finishes; cluster remains until stopped or deleted
```

A successful job does not automatically delete an existing cluster. A failed job also does not necessarily stop its compute costs.

For scheduled pipelines, orchestration handles dependencies: wait for source files, run transformations, load analytical tables, and handle failures. Workflow templates or an orchestrator can coordinate these steps.

Keep output paths separate for different runs, or design a deliberate replacement strategy. Blindly retrying append operations can duplicate data.

## 10. A small PySpark transformation

Save this illustrative script as `product_sales.py`. It accepts an input path and output path as arguments. The CSV columns are `order_id`, `product`, and `amount`.

```python
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

input_path, output_path = sys.argv[1:3]
spark = SparkSession.builder.appName("product-sales").getOrCreate()

orders = (
    spark.read
    .option("header", True)
    .schema("order_id STRING, product STRING, amount DECIMAL(12,2)")
    .csv(input_path)
)

clean_orders = orders.filter(
    F.col("order_id").isNotNull()
    & F.col("product").isNotNull()
    & F.col("amount").isNotNull()
    & (F.col("amount") >= 0)
)

product_sales = clean_orders.groupBy("product").agg(
    F.sum("amount").alias("total_sales"),
    F.count("*").alias("order_count"),
)

product_sales.write.mode("errorifexists").parquet(output_path)
spark.stop()
```

The explicit schema gives the amount a numeric type. Invalid amounts become null under the default permissive CSV behavior and are filtered out. In a production pipeline, route rejected rows to a separate output when they need investigation.

This example counts input rows as orders; it assumes one row per order and does not resolve duplicate order IDs. The output mode fails if the destination already exists, making replacement a deliberate decision.

## 11. How the script is submitted

After a cluster, IAM access, network, and input files are configured, upload the script and submit it from an Ubuntu terminal. Replace every placeholder.

```bash
gcloud storage cp product_sales.py gs://YOUR_BUCKET/scripts/product_sales.py

gcloud dataproc jobs submit pyspark gs://YOUR_BUCKET/scripts/product_sales.py \
  --project=YOUR_PROJECT_ID \
  --region=YOUR_REGION \
  --cluster=YOUR_CLUSTER \
  -- \
  gs://YOUR_BUCKET/raw/orders/ \
  gs://YOUR_BUCKET/curated/product_sales/run-001/
```

Arguments after `--` go to the Python script. The script reads them as `input_path` and `output_path`.

In the Console, the equivalent workflow is to open the Dataproc jobs area, submit a **PySpark** job, select a cluster and region, provide the main Python file URI, and enter its arguments. Labels can vary with the Managed Spark naming.

Submitting from dataengenv sends work to the remote cluster; it does not make your Ubuntu machine a worker.

[Submitting jobs](https://docs.cloud.google.com/managed-spark/docs/guides/submit-job)

## 12. Scaling and cost decisions

**Manual scaling** changes worker capacity explicitly. **Autoscaling** uses a configured policy to adjust eligible worker capacity within limits. It does not rewrite inefficient Spark code.

Cluster autoscaling and Spark dynamic allocation solve different problems: autoscaling adjusts machines, while dynamic allocation adjusts executor counts within available resources. They must be configured compatibly.

For predictable batch jobs, a temporary cluster can reduce idle running time. For frequent jobs, a shared cluster may reduce repeated startup overhead but requires capacity management.

Compute, disks, storage, service charges, and data transfer can contribute to cost. Stopping a cluster does not remove every charge because persistent resources can remain. Deleting a cluster removes its compute lifecycle; GCS data remains separately managed.

Avoid choosing worker counts only by file size. A large shuffle, data skew, or many small files can dominate runtime.

[Cluster autoscaling](https://docs.cloud.google.com/managed-spark/docs/concepts/configuring-clusters/autoscaling)

## 13. Understand common problems

| Symptom | Likely area to investigate |
|---|---|
| Cluster cannot be created | Quota, region policy, machine availability, network, or IAM |
| Job cannot read a GCS path | Runtime service-account permissions or incorrect object path |
| Python package cannot be imported | Dependency is missing from the remote runtime |
| Executors run out of memory | Large partitions, joins, caching, or skew |
| Most tasks finish but a few take much longer | Uneven data distribution or unusually large partitions |
| Output path already exists | Previous output plus the selected write mode |
| Driver crashes after collecting data | Too much distributed data brought into driver memory |

Use job details and driver logs for application errors. Executor logs and the Spark UI help explain task failures, shuffles, and uneven execution. Keep large results in distributed storage instead of calling `collect()` on the entire dataset.

## 14. Choose the service for the work

| Requirement | Starting choice | Reason |
|---|---|---|
| Existing PySpark pipeline with custom cluster needs | Dataproc cluster | Control over the Spark environment |
| Independent supported Spark batch | Serverless Spark | Managed execution without maintaining a cluster |
| Transformations expressed cleanly in analytical SQL | BigQuery | Keep processing close to analytical tables |
| Apache Beam batch or event-time streaming pipeline | Dataflow | Managed Beam execution |
| Upload, download, or list objects | GCS Python client | No distributed processing required |

Dataproc can also run Spark streaming workloads; streaming is not exclusive to Dataflow. Choose based on the processing framework and operating requirements.

For the retail example, start with **GCS input ? PySpark transformation ? GCS output**. Add BigQuery when the curated data needs SQL reporting. Add orchestration when the flow must run repeatedly with dependencies.